# CSV + API

In this reboot, we are going to use:

- The [Goodreads books](https://www.kaggle.com/jealousleopard/goodreadsbooks) dataset from Kaggle.
- The [Open Library Books API](https://openlibrary.org/dev/docs/api/books)

The goal of this livecode is to load the data from a CSV + loop over rows to enrich each row with information such as:

- List of subjects (Science, Humor, Travel, etc.)
- The cover URL of the book
- Other information you'd find useful in the JSON API

First, download the CSV in the local folder:

In [42]:
!curl -L https://wagon-public-datasets.s3.amazonaws.com/02-Data-Toolkit/02-Data-Sourcing/books.csv > books.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1509k  100 1509k    0     0   577k      0  0:00:02  0:00:02 --:--:--  577k


In [43]:
!ls -lh

total 4296
-rw-r--r--@ 1 scooter  staff   1.5M Oct  8 17:37 books.csv
-rw-r--r--@ 1 scooter  staff   579B Jan 18  2026 README.md
-rw-r--r--@ 1 scooter  staff    31K Oct  8 17:36 Recap.ipynb


Then import the usual suspects!

In [44]:
import requests
import pandas as pd
import numpy as np

## Load books from CSV

In [45]:
books_df = pd.read_csv('books.csv')
books_df.head(10)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964
5,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling-Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,38872,154
6,9,"Unauthorized Harry Potter Book Seven News: ""Ha...",W. Frederick Zimmerman,3.69,0976540606,9780976540601,en-US,152,18,1
7,10,Harry Potter Collection (Harry Potter #1-6),J.K. Rowling,4.73,0439827604,9780439827607,eng,3342,27410,820
8,12,The Ultimate Hitchhiker's Guide: Five Complete...,Douglas Adams,4.38,0517226952,9780517226957,eng,815,3602,258
9,13,The Ultimate Hitchhiker's Guide to the Galaxy,Douglas Adams,4.38,0345453743,9780345453747,eng,815,240189,3954


In [22]:
# url = "https://openlibrary.org/api/books?bibkeys=ISBN:0439785960&format=json&jscmd=data"
# requests.get(url).json()

# Make a function that would take a single ISBN, and returns a url for the cover of the book
def cover_from_isbn(isbn):
    # construct the url
    url = f"https://openlibrary.org/api/books?bibkeys=ISBN:{isbn}&format=json&jscmd=data"
    # get the json from the API
    data = requests.get(url).json()
    # dig into the dict to get the cover url
    return data[f"ISBN:{isbn}"]['cover']['large']

In [46]:
# Iterate over the first 10 rows and call the function cover_from_isbn on each book
for index, book in books_df.head(10).iterrows():
    cover_url = cover_from_isbn(book["isbn"])
    # set the url as a new column in the dataframe `cover_url`
    books_df.loc[index, 'cover_url'] = cover_url

In [51]:
books_df.head(20)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,# num_pages,ratings_count,text_reviews_count,cover_url
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling-Mary GrandPré,4.56,0439785960,9780439785969,eng,652,1944099,26249,https://covers.openlibrary.org/b/id/15156081-L...
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling-Mary GrandPré,4.49,0439358078,9780439358071,eng,870,1996446,27613,https://covers.openlibrary.org/b/id/12025650-L...
2,3,Harry Potter and the Sorcerer's Stone (Harry P...,J.K. Rowling-Mary GrandPré,4.47,0439554934,9780439554930,eng,320,5629932,70390,https://covers.openlibrary.org/b/id/7572543-L.jpg
3,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.41,0439554896,9780439554893,eng,352,6267,272,https://covers.openlibrary.org/b/id/10301720-L...
4,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling-Mary GrandPré,4.55,043965548X,9780439655484,eng,435,2149872,33964,https://covers.openlibrary.org/b/id/8778528-L.jpg
5,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling-Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,38872,154,https://covers.openlibrary.org/b/id/278981-L.jpg
6,9,"Unauthorized Harry Potter Book Seven News: ""Ha...",W. Frederick Zimmerman,3.69,0976540606,9780976540601,en-US,152,18,1,https://covers.openlibrary.org/b/id/742235-L.jpg
7,10,Harry Potter Collection (Harry Potter #1-6),J.K. Rowling,4.73,0439827604,9780439827607,eng,3342,27410,820,https://covers.openlibrary.org/b/id/279436-L.jpg
8,12,The Ultimate Hitchhiker's Guide: Five Complete...,Douglas Adams,4.38,0517226952,9780517226957,eng,815,3602,258,https://covers.openlibrary.org/b/id/12617870-L...
9,13,The Ultimate Hitchhiker's Guide to the Galaxy,Douglas Adams,4.38,0345453743,9780345453747,eng,815,240189,3954,https://covers.openlibrary.org/b/id/14656530-L...


Let's add a new column

In [ ]:
# YOUR CODE HERE

## API - Open Library

In [ ]:
# YOUR CODE HERE

## Calling the API with multiple ISBNs at a time

In [48]:
# Get all the cover URL for every single books in the df
# Think of how we can make the query parameters where 50 ISBNs are combined together
# Think of how we batch process the response back from the API

In [76]:
# Make a function that takes a list of ISBNs, and then returns (list / dict) of cover URLs
# https://openlibrary.org/api/books?bibkeys=ISBN:0385472579,ISBN:62019420&format=json

def covers_from_isbns(isbns):
  # build the query param for bibkeys
  bibkeys = f"ISBN:{",ISBN:".join(isbns)}"
  # build the url
  url = f"https://openlibrary.org/api/books?bibkeys={bibkeys}&format=json&jscmd=data"

  data = requests.get(url).json()

  # create a return list
  cover_urls = []
  # put all the cover url into the list
  for key in data.keys():
    cover_urls.append(data[key]['cover']['large'])
  # and return
  return cover_urls

In [85]:
isbns = list(books_df['isbn'])
isbns

['0439785960',
 '0439358078',
 '0439554934',
 '0439554896',
 '043965548X',
 '0439682584',
 '0976540606',
 '0439827604',
 '0517226952',
 '0345453743',
 '1400052920',
 '0739322206',
 '0517149257',
 '076790818X',
 '0767915062',
 '0767910435',
 '0767903862',
 '076790382X',
 '0060920084',
 '0380713802',
 '0380727501',
 '0380715430',
 '0345538374',
 '0618517650',
 '0618346244',
 '0618346252',
 '0618260587',
 '0618391002',
 '0618510826',
 '0618153977',
 '193337201X',
 '097669400X',
 '0689840926',
 '1557344493',
 '0385326505',
 '1575606240',
 '1595580271',
 '1595962808',
 '0670059676',
 '0141312629',
 '0595321801',
 '1590301943',
 '0449146979',
 '0061159174',
 '006076273X',
 '0060749911',
 '0273704745',
 '1932386106',
 '096513671X',
 '0374517193',
 '0374280398',
 '0374519749',
 '0374522596',
 '0374518734',
 '0374522871',
 '0374519323',
 '0374516006',
 '0374520658',
 '0822205106',
 '0679734996',
 '1596670231',
 '1581807740',
 '0800793617',
 '0871202867',
 '0753454947',
 '0871209993',
 '18524028

In [ ]:
final_list_of_urls = []

for small_isbns in np.array_split(isbns, 260):
    urls = covers_from_isbns(small_isbns)
    final_list_of_urls.append(urls)
    print(final_list_of_urls)

final_list_of_urls
# urls = covers_from_isbns(isbns)

KeyError: 'cover'

0     https://covers.openlibrary.org/b/id/15156081-L...
1     https://covers.openlibrary.org/b/id/12025650-L...
2     https://covers.openlibrary.org/b/id/7572543-L.jpg
3     https://covers.openlibrary.org/b/id/10301720-L...
4     https://covers.openlibrary.org/b/id/8778528-L.jpg
5      https://covers.openlibrary.org/b/id/278981-L.jpg
6      https://covers.openlibrary.org/b/id/742235-L.jpg
7      https://covers.openlibrary.org/b/id/279436-L.jpg
8     https://covers.openlibrary.org/b/id/12617870-L...
9     https://covers.openlibrary.org/b/id/14656530-L...
10                                                  NaN
11                                                  NaN
12                                                  NaN
13                                                  NaN
14                                                  NaN
15                                                  NaN
16                                                  NaN
17                                              